# 05 · Does hyperspectral data add value? (EnMAP vs Sentinel-2)

Our documented limitation: with Sentinel-2, a single index (BSI) separates built surfaces from bright desert soil at only ~75%. This notebook tests whether EnMAP's ~200 usable bands do better, **with everything else held equal**: same 30 m pixels, same labels, same classifier, same spatial cross-validation (1 km tiles held out together).

Labels: *built* = 30 m pixels ≥ 90% covered by OSM footprints; *open* = > 300 m from any footprint and not vegetated.

EnMAP scene: 22 Apr 2025 (see `config/enmap_scenes.txt`). *Contains modified EnMAP data © DLR [2025].* Raw EnMAP data is not stored in this repository (licence).

In [1]:
from coolcairo.config import load_config, geobox_for
from coolcairo.pipeline import load_enmap, load_sentinel2, load_osm_buildings, FINE_RESOLUTION
from coolcairo.buildings import footprint_mask
from coolcairo.classify import bare_soil_index
from coolcairo import enmap, separability
cfg = load_config()
refl = load_enmap(cfg)
s2_30 = load_sentinel2(cfg).coarsen(x=3, y=3, boundary='trim').mean().assign_coords(x=refl.x, y=refl.y)
roof10 = footprint_mask(load_osm_buildings(cfg), geobox_for(cfg, 'model_aoi', FINE_RESOLUTION)).values
ndvi30 = ((s2_30.B08 - s2_30.B04) / (s2_30.B08 + s2_30.B04)).values
labels = separability.labels_30m(roof10, ndvi30, cfg['s2_classification']['ndvi_vegetation'])
print(refl.sizes, {'built': int((labels == 1).sum()), 'open': int((labels == 0).sum())})

Frozen({'wavelength': 203, 'y': 453, 'x': 459}) {'built': 1211, 'open': 78441}


## Head-to-head
*EnMAP simulated as Sentinel-2* averages EnMAP bands inside each Sentinel-2 passband: same pixels and date, only spectral detail removed. Comparing it with *EnMAP full spectrum* isolates the value of hyperspectral resolution.

In [2]:
s2_bands = list(enmap.S2_PASSBANDS_NM)
table = separability.compare({
    'Sentinel-2 BSI only (old rule)': bare_soil_index(s2_30).expand_dims(feature=['BSI']),
    'Sentinel-2 real, 6 bands': s2_30[s2_bands].to_array('feature'),
    'EnMAP simulated as Sentinel-2, 6 bands': enmap.simulate_sentinel2(refl).to_array('feature'),
    'EnMAP full spectrum': refl.rename(wavelength='feature'),
}, labels, refl.x.values, refl.y.values, cfg['cv_tile_size'], cfg['cv_folds'])
table.round(3)

,features,n_features,balanced_accuracy,std_across_folds
0,Sentinel-2 BSI only (old rule),1,0.760,0.037
1,"Sentinel-2 real, 6 bands",6,0.860,0.022
2,"EnMAP simulated as Sentinel-2, 6 bands",6,0.797,0.032
3,EnMAP full spectrum,203,0.816,0.030


## Does EnMAP add anything on top of the Sentinel-2 composite?

In [3]:
base, fused, std = separability.fused_accuracy(
    s2_30[s2_bands].to_array('feature'), refl, labels, refl.x.values, refl.y.values,
    cfg['cv_tile_size'], cfg['cv_folds'])
print(f'Sentinel-2 alone {base:.3f}  |  Sentinel-2 + EnMAP {fused:.3f} (fold std {std:.3f})')

Sentinel-2 alone 0.860  |  Sentinel-2 + EnMAP 0.862 (fold std 0.026)


## Test 2: does EnMAP explain surface heat better?
Block-mean spectra (90 m) predict block summer LST with ridge regression, spatial CV. Urban blocks only (≥ 10% building coverage). Also shown with larger held-out tiles (2 and 3 km) to guard against spatial leakage.

In [4]:
import numpy as np
import pandas as pd
from coolcairo.pipeline import build_blocks
from coolcairo.model import FEATURES, spatial_groups
blocks, _, _ = build_blocks(cfg)
urban = blocks.dropna(subset=['lst_c', *FEATURES])
urban = urban[urban.roof_frac >= cfg['min_building_coverage']]
s2_10 = load_sentinel2(cfg)[s2_bands].to_array('feature')
ours = urban[FEATURES].to_numpy()
spec = {
    'Sentinel-2 spectra (6)': separability.block_spectra(s2_10, 9, urban),
    'EnMAP simulated as S2 (6)': separability.block_spectra(enmap.simulate_sentinel2(refl).to_array('feature'), 3, urban),
    'EnMAP full spectrum': separability.block_spectra(refl, 3, urban),
}
sets = {'Our model features': ours, **spec,
        'Ours + Sentinel-2': np.hstack([ours, spec['Sentinel-2 spectra (6)']]),
        'Ours + EnMAP': np.hstack([ours, spec['EnMAP full spectrum']])}
pd.DataFrame({f'{t} m tiles': separability.heat_r2(sets, urban.lst_c.to_numpy(), spatial_groups(urban, t), cfg['cv_folds'])
              for t in (1000, 2000, 3000)}).round(3)

,1000 m tiles,2000 m tiles,3000 m tiles
Our model features,0.251,0.342,0.259
Sentinel-2 spectra (6),0.312,0.327,0.328
EnMAP simulated as S2 (6),0.331,0.353,0.351
EnMAP full spectrum,0.534,0.468,0.455
Ours + Sentinel-2,0.402,0.444,0.400
Ours + EnMAP,0.583,0.552,0.501


## Conclusion (30 Sep 2026 run)
- Hyperspectral detail carries real extra information: full EnMAP beats the same scene reduced to 6 bands by ~2–4 points (robust across logistic regression and random forest).
- For built-vs-desert separation, a 70-scene Sentinel-2 summer composite already captures it: fusing EnMAP adds ~0.2 points (within fold noise).
- A 6-band Sentinel-2 classifier reaches ~86% on these clean 30 m test pixels, but it does **not** transfer to the full map: at 10 m it scores ~76% (same as the BSI rule), and using it in the pipeline (at 10 m or 30 m) lowered the heat model's spatial-CV R² from 0.25 to 0.20–0.22. The pipeline therefore keeps the simple BSI rule.
- **For explaining surface heat, hyperspectral clearly adds value** (test 2): EnMAP block spectra reach spatial-CV R² ≈ 0.46–0.53 vs 0.35 for the same scene reduced to Sentinel-2 bands, and our model features + EnMAP reach 0.50–0.58 vs 0.40–0.44 with Sentinel-2. Robust to 1–3 km held-out tiles.
- (Shadow was ruled out as the cause of the dark-roof sign: at 65–75° sun elevation a 16 m building casts a ~6 m shadow, a fraction of a 30 m pixel.)